# Transforming a RAG pipeline into an AI Agent

In [12]:
%pip install -q openai qdrant-client docling fastembed jupyter-chat-widget

In [13]:
# Chat UI example

from time import sleep
from jupyter_chat_widget import ChatUI

chat = ChatUI()

def example(msg):
    chat.rewrite("Thinking...")
    sleep(1)
    chat.rewrite("")
    for word in msg.split(" "):
        chat.append(word + " ")
        sleep(0.5)

chat.connect(example)

Output()

Output()

Text(value='', description='user: ')

# Documents

let's download some files that we will use for RAG later.

In [14]:
import os
import requests

# Add all the files you want in your RAG pipeline
# This example uses a snippet from Claude Sonnet's 4.5 System Card, specifically the Introduction chapter.
urls_and_filenames = [
    ("https://gist.githubusercontent.com/ZanSara/919f8071f8d977cbc1885af334ff6616/raw/461d20496deb1326c4c4e3021883a5a861d40c16/claude_sonnet_4.5_system_card-ch1.md", "sonnet-4.5.md"),
]

if not os.path.exists("documents"):
    os.makedirs("documents")

for url, filename in urls_and_filenames:
    response = requests.get(url)
    response.raise_for_status()
    with open("documents/" + filename, "wb") as f:
        f.write(response.content)

# Basic chatbot (no RAG)

 Build a very simple LLM chatbot with no RAG nor agentic skills. A simple function that calls an LLM and prints out the response.

In [15]:
import openai
import os
from google.colab import userdata


def get_oai_client():
    """
        Creates the OpenAI client.
    """
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    client = openai.OpenAI()
    messages = [{"role": "system", "content": "You are a helpful assistant. Output in HTML."}]
    return client, messages


def query_llm(ui, client, messages):
    """
        Query the LLM and returns a stream handle to the response.
    """
    ui.rewrite("[Generating]")
    stream = client.chat.completions.create(
        model="gpt-5.2",  # Or another suitable model
        messages=messages, # Use the message history
        stream=True, # Enable streaming
    )
    return stream


def display_response(ui, stream):
    """
        Display the LLM's response in the UI one token at a time.
    """
    ui.rewrite("")
    complete_response = ""
    for chunk in stream:
        if (content := chunk.choices[0].delta.content) and content.strip():
            complete_response += content
            ui.append(content)
    return complete_response

In [16]:
def chat():
    """
        Chat with an LLM directly (no RAG)
    """
    ui = ChatUI()
    client, messages = get_oai_client()

    # This method is called every time the user enters a new message
    def _chat(query):
        messages.append({"role": "user", "content": query})
        stream = query_llm(ui, client, messages)
        response = display_response(ui, stream)
        messages.append({"role": "assistant", "content": response})

    ui.connect(lambda query: _chat(query))

In [17]:
# Did Sonnet 4.5 cross the ASL-4 threshold?
chat()

Output()

Output()

Text(value='', description='user: ')

# RAG

In [18]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter
from uuid import uuid4
from qdrant_client import QdrantClient, models


def ingest_documents(ui, paths):
    # Prepare the vectordb and embedder
    vdb = QdrantClient(location=":memory:")
    dense_model = "sentence-transformers/all-MiniLM-L6-v2"
    vdb.set_model(dense_model)
    collection_name = "documents"
    vdb.create_collection(
        collection_name=collection_name,
        vectors_config=vdb.get_fastembed_vector_params(),
    )
    points = []
    for path in paths:
        ui.rewrite(f"[Parsing {path}...]")

        # Parse the document with docling
        doc = DocumentConverter().convert(source=path).document

        # Chunk the document
        chunker = HybridChunker()
        chunk_iter = chunker.chunk(dl_doc=doc)

        # Enrich the chunks and build Qdrant points
        for chunk in chunk_iter:
            enriched_text = chunker.contextualize(chunk=chunk)
            meta = chunk.meta.export_json_dict()
            points.append(
                models.PointStruct(
                    id=uuid4().hex,
                    payload=meta | {"document": enriched_text},
                    vector={
                        # FastEmbed uses named vector fields derived from the model names
                        vdb.get_vector_field_name(): models.Document(text=enriched_text, model=dense_model),
                    },
                )
            )
    # Upload (embeddings happen internally because we used models.Document)
    vdb.upload_points(collection_name=collection_name, points=points, batch_size=64, wait=True)
    ui.rewrite(f"All documents were processed. I'm ready!")
    return vdb

def add_context(ui, vdb, query):
    """
        Queries the vector database for relevant context snippets
        and adds them to the LLM's context.
    """
    ui.rewrite("[Searching]")
    samples = vdb.query(
        collection_name="documents",
        query_text=query,
        limit=10,
    )
    ui.rewrite(f"[Found {len(samples)} relevant snippets]")
    sleep(1)
    return {
        "role": "user",
        "content": f"Relevant snippets from the document: {'\n\n'.join(s.document for s in samples)}"
    }

In [19]:
def basic_rag(paths):
    """
        Basic RAG on a set of documents.
    """
    ui = ChatUI()
    vdb = ingest_documents(ui, paths)
    client, messages = get_oai_client()

    def _basic_rag(query):
        messages.append({"role": "user", "content": query})
        messages.append(add_context(ui, vdb, query))
        stream = query_llm(ui, client, messages)
        response = display_response(ui, stream)
        messages.append({"role": "assistant", "content": response})

    ui.connect(lambda query: _basic_rag(query))

In [20]:
# Did Sonnet 4.5 cross the ASL-4 threshold?
basic_rag(["/content/documents/sonnet-4.5.md"])

Output()

Output()

Text(value='', description='user: ')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

In [21]:
def rewrite_query(ui, client, messages, query):
    ui.rewrite("[Rewriting query...]")
    response = client.chat.completions.create(
        model="gpt-5.2",  # Or another suitable model
        messages=[{
            "role": "system",
            "content": "Read the following chat and write an input text for a vector database that would retrieve the correct context. Keep it concise and don't add anything that the user didn't ask for. For example if the user asked 'How many years can a turtle live' and the current question is 'what about spiders?', you should not modify the first question, while you should modify the second one to read 'how many years do spiders live?'"
            }] + messages,
    )
    query = response.choices[0].message.content
    ui.rewrite(f"[New query: '{query}']")
    sleep(2)  # Just to be able to read it!
    return query

In [22]:
def rag_with_query_rewrite(paths):
    ui = ChatUI()
    vdb = ingest_documents(ui, paths)
    client, messages = get_oai_client()

    def _rag_with_query_rewrite(query):
        messages.append({"role": "user", "content": query})
        query = rewrite_query(ui, client, messages, query)
        messages.append(add_context(ui, vdb, query))
        stream = query_llm(ui, client, messages)
        response = display_response(ui, stream)
        messages.append({"role": "assistant", "content": response})

    ui.connect(lambda query: _rag_with_query_rewrite(query))

In [23]:
# Did Sonnet 4.5 cross the ASL-4 threshold?
# How about other models?
rag_with_query_rewrite(["/content/documents/sonnet-4.5.md"])

Output()

Output()

Text(value='', description='user: ')

In [24]:
def retrieval_decision(ui, client, messages, query):
    ui.rewrite("[Checking if retrieval is needed...]")
    response = client.chat.completions.create(
        model="gpt-5.2",  # Or another suitable model
        messages=[{
            "role": "system",
            "content": "You're a step in a RAG pipeline and your task is to decide whether to use the retriever or not. Answer only YES if we should retrieve, NO otherwise. Don't output any other text. Don't perform any other action. YOUR ANSWER MUST BE YES OR NO EVERY TIME."
            }] + messages,
    )
    decision = response.choices[0].message.content
    ui.rewrite(f"[Decision: '{decision}']")
    sleep(1)
    return True if decision.lower() == "yes" else False

In [25]:
def rag_with_query_rewrite_and_optional_retrieval(paths):
    ui = ChatUI()
    vdb = ingest_documents(ui, paths)
    client, messages = get_oai_client()

    def _rag_with_query_rewrite_and_optional_retrieval(query):
        messages.append({"role": "user", "content": query})
        decision = retrieval_decision(ui, client, messages, query)
        if decision:
            query = rewrite_query(ui, client, messages, query)
            messages.append(add_context(ui, vdb, query))
        stream = query_llm(ui, client, messages)
        response = display_response(ui, stream)
        messages.append({"role": "assistant", "content": response})

    ui.connect(lambda query: _rag_with_query_rewrite_and_optional_retrieval(query))

In [26]:
# Did Sonnet 4.5 cross the ASL-4 threshold?
# Translate this answer in French
rag_with_query_rewrite_and_optional_retrieval(["/content/documents/sonnet-4.5.md"])

Output()

Output()

Text(value='', description='user: ')

In [27]:
def agentic_rag(paths):
    ui = ChatUI()
    client, messages = get_oai_client()
    vdb = ingest_documents(ui, paths)

    def _agentic_rag(query):
        messages.append({"role": "user", "content": query})
        while retrieval_decision(ui, client, messages, query):
            query = rewrite_query(ui, client, messages, query)
            messages.append(add_context(ui, vdb, query))
        stream = query_llm(ui, client, messages)
        response = display_response(ui, stream)
        messages.append({"role": "assistant", "content": response})

    ui.connect(lambda query: _agentic_rag(query))

In [32]:
# What AI Safety Level was Sonnet released under, and why?
agentic_rag(["/content/documents/sonnet-4.5.md"])

Output()

Output()

Text(value='', description='user: ')

In [34]:
import json

def _invoke_tool(ui, client, vdb, messages, tools, first_token, stream):
    # Collect the tool's data
    tool_call = first_token.choices[0].delta.tool_calls[0]
    tool_name = tool_call.function.name
    ui.rewrite(f"Calling tool {tool_name}")

    # Collect the parameters
    args_json = ""
    for token in stream:
        if token.choices[0].delta.tool_calls:
            args_json += token.choices[0].delta.tool_calls[0].function.arguments
    tool_args = json.loads(args_json)

    # Invoke the tool
    tool_implementation = tool_implementations[tool_name]
    output = tool_implementation(ui, vdb, **tool_args)

    # Add the output to the messages
    messages.append(output)

    # call the LLM again
    return query_llm_with_tools(ui, client, vdb, messages, tools)


def query_llm_with_tools(ui, client, vdb, messages, tools):
    """
        Query the LLM and returns a stream handle to the response.
        If the LLM is trying to invoke tools, it will be done here
        in a loop until the LLM decides to reply to the user.
    """
    ui.rewrite("[Generating]")
    stream = client.chat.completions.create(
        model="gpt-5.2",  # Or another suitable model
        messages=messages, # Use the message history
        stream=True, # Enable streaming
        tools=tools
    )
    first_token = next(stream)
    if first_token.choices[0].delta.tool_calls:
        return _invoke_tool(ui, client, vdb, messages, tools, first_token, stream)
    else:
        return stream

In [35]:
tool_implementations = {
    "search": add_context
}

search = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Use this tool when the user is asking you something you don't know. You should use this tool very often and you can call it many times in a row if necessary.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description":
                        "The query to retrieve the information from the document store using pure embedding similarity search",
                },
            },
            "required": ["query"],
        },
    }
}

def retrieval_as_tool(paths):
    ui = ChatUI()
    vdb = ingest_documents(ui, paths)
    client, messages = get_oai_client()

    def _retrieval_as_tool(query):
        messages.append({"role": "user", "content": query})
        stream = query_llm_with_tools(ui, client, vdb, messages, [search])
        response = display_response(ui, stream)
        messages.append({"role": "assistant", "content": response})

    ui.connect(lambda query: _retrieval_as_tool(query))

In [36]:
# Did Sonnet 4.5 cross the ASL-4 threshold?
retrieval_as_tool(["/content/documents/sonnet-4.5.md"])

Output()

Output()

Text(value='', description='user: ')